# From chains to graphs to swarms: PierPoint Release Desk in LangGraph

**Stack:** LangChain 1.x, LangGraph 1.x, Amazon Bedrock. Python 3.11+, runs in VS Code.

You have not seen LangGraph before. That is fine, because this notebook does not start with LangGraph. It starts with a plain chain, hits a wall, and only then reaches for the graph. The point is that you should feel the wall before you are handed the ladder.

Here is the shape of the journey.

| Part | What you build | What you learn |
|---|---|---|
| 1 | The scenario | Why one workflow needs two different tools |
| 2 | A LangChain chain | LCEL, and exactly where a chain runs out of road |
| 3 | Your first StateGraph | State, nodes, edges, reducers |
| 4 | The same graph, with controls | Checkpoints, pause and resume, caps, retries, time travel |
| 5 | Parallel work | Fan-out, reducers under concurrency, fan-out decided at runtime |
| 6 | LangGraph's two jobs | Runtime for agents, and state machine for workflows |
| 7 | Swarms, three ways | Hand-rolled handoff, the prebuilt library, and a supervisor |
| 8 | The decision | Graph or swarm, with a rule you can apply in code review |
| 9 | The hybrid | A swarm living inside a control graph |
| 10 | Production | Threads, durability, observability, a test suite with no model calls |

Every code cell runs. The ones that call Bedrock are marked. Roughly two thirds of the notebook runs with no AWS credentials at all, because most of what LangGraph does is not model work.

**A note on the diagrams.** Where a picture helps, there is one. LangGraph can also draw itself, and from Part 3 onward you will be generating the diagrams straight from the code instead of drawing them by hand. If Mermaid blocks show as plain text in VS Code, install the extension `bierner.markdown-mermaid`.

In [ ]:
# Install once, then restart the kernel.
# pip install -U langgraph langchain langchain-aws langgraph-swarm boto3 pydantic

import importlib.metadata as md

for pkg in ("langgraph", "langchain", "langchain-core", "langchain-aws", "langgraph-swarm"):
    try:
        print(f"{pkg:<20} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:<20} MISSING")

# This notebook was written and verified against langgraph 1.2.x and langchain 1.3.x.
# The v1 line moved a few things. Two you will hit immediately if you follow older tutorials:
#   create_react_agent  ->  from langchain.agents import create_agent
#   MemorySaver         ->  InMemorySaver  (the old name still imports)

In [ ]:
import json
import operator
import time
import uuid
from typing import Annotated, Any, Literal, TypedDict

from pydantic import BaseModel, Field

from langchain_aws import ChatBedrockConverse
from langchain_core.messages import ToolMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableParallel
from langchain_core.tools import InjectedToolCallId, tool

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, MessagesState, StateGraph
from langchain.agents import create_agent
from langgraph.prebuilt import InjectedState
from langgraph.types import Command, RetryPolicy, Send, interrupt

import os

REGION = os.environ.get("AWS_REGION", "us-east-1")

# Claude on-demand needs the cross-region inference profile prefix. Nova does not.
REASONING_MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
CHEAP_MODEL = "amazon.nova-lite-v1:0"


def chat(model_id: str = REASONING_MODEL, temperature: float = 0.2, max_tokens: int = 900) -> ChatBedrockConverse:
    """One place where models are built. Set temperature only: Claude 4.x rejects
    temperature and top_p together."""
    return ChatBedrockConverse(
        model_id=model_id,
        region_name=REGION,
        temperature=temperature,
        max_tokens=max_tokens,
    )


print("region:", REGION)

In [ ]:
# Preflight. Do this now so a credentials problem does not surface twenty cells later.
try:
    reply = chat(REASONING_MODEL, temperature=0.0, max_tokens=16).invoke("Reply with one word: READY")
    print("bedrock:", reply.content)
    BEDROCK_OK = True
except Exception as exc:
    BEDROCK_OK = False
    print(f"bedrock unavailable: {type(exc).__name__}: {str(exc)[:180]}")
    print("\nCells marked [needs bedrock] will fail. Everything else still runs.")

# AccessDeniedException  -> model not enabled, or the IAM policy is missing bedrock:InvokeModel
# ValidationException    -> the "us." inference profile prefix is missing on a Claude model
# ThrottlingException    -> regional capacity, try again or drop to the cheap model

## 1. The scenario

A container is stuck at PierPoint terminal and the customer wants it released. Nobody knows yet why it is stuck.

This is a good teaching case because the work splits cleanly into two kinds, and most real enterprise workflows do the same thing without anyone noticing.

**The first kind is a contract.** Validate the request, plan the fix, get approval if money is involved, apply the effects, write the audit record. You could print these steps on a laminated card and hand them to a new joiner. The order matters, and getting it wrong has consequences an auditor will ask about.

**The second kind is a huddle.** Finding out *why* the container is stuck. Maybe customs, maybe billing, maybe damage, maybe a broken reefer plug, maybe two of those at once. You cannot write the sequence down in advance because the sequence depends on what the last check turned up.

Five sentences an auditor would read back to you:

| # | Invariant |
|---|---|
| I1 | A release is issued at most once per container per request |
| I2 | A fee waiver above the threshold needs a named human approver |
| I3 | Diagnosis is read-only. Nothing that investigates may write |
| I4 | Every customer message traces back to the findings behind it |
| I5 | A run that hits a cap ends reviewable, never half-applied |

And the six stages. Look only at the third column.

| # | Stage | Can you name the next step in advance? | Writes? |
|---|---|---|---|
| 1 | Validate the request and the requesting party | Yes | No |
| 2 | Find out why the container is blocked | **No** | No |
| 3 | Turn findings into a remediation plan | Yes | No |
| 4 | Approve if exposure crosses the threshold | Yes | No |
| 5 | Apply effects: waive, release, notify | Yes | **Yes** |
| 6 | Write the audit record | Yes | Yes |

One row out of six says no. That single row is the entire case for a swarm, and the other five are the entire case against using one anywhere else. Hold on to that, because in Part 8 it becomes a rule.

## 2. Level one: a LangChain chain

Start with the simplest thing that could work.

LangChain gives you the pipe operator. You take a prompt, a model, and a parser, and you join them with `|`. The result is a Runnable, which is just an object with `.invoke()`, `.stream()` and `.batch()` on it. Everything in LangChain speaks that interface, which is why you can pipe them together at all.

```mermaid
flowchart LR
    IN["input dict"] --> P["ChatPromptTemplate"]
    P --> M["ChatBedrockConverse"]
    M --> O["StrOutputParser"]
    O --> OUT["string"]
```

The whole thing is one arrow. Data goes in the left, comes out the right, nothing loops back. That property is worth naming now, because it is what eventually breaks.

In [ ]:
# [needs bedrock]
diagnose_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a terminal operations analyst. Two sentences. State the blocker only."),
    ("human", "Container {container_id}. Customs: {customs}. Billing: {billing}. Why is it not moving?"),
])

diagnose_chain = diagnose_prompt | chat(REASONING_MODEL) | StrOutputParser()

CASE = {
    "container_id": "MSCU7391045",
    "customs": "documentary hold, invoice value mismatch, not cleared",
    "billing": "6 days demurrage, 1800 USD accrued, dispute open",
}

if BEDROCK_OK:
    print(diagnose_chain.invoke(CASE))
else:
    print("skipped, no bedrock")

That works, and for a surprising number of production tasks that really is the whole answer. Do not let anyone talk you into a graph when a chain does the job.

LangChain also has more than one shape of arrow. Three worth knowing before we move on, all of which run without a model.

In [ ]:
# Runs offline. Three LCEL shapes beyond the straight pipe.

# 1. Fan out to several branches at once, get a dict back.
checks = RunnableParallel(
    customs=RunnableLambda(lambda x: f"customs: hold on {x['container_id']}"),
    billing=RunnableLambda(lambda x: f"billing: 1800 usd on {x['container_id']}"),
)
print("parallel :", checks.invoke({"container_id": "MSCU7391045"}))

# 2. Pick a branch based on the input.
approval_branch = RunnableBranch(
    (lambda x: x["exposure"] > 500, RunnableLambda(lambda x: "NEEDS APPROVAL")),
    RunnableLambda(lambda x: "AUTO APPROVE"),
)
print("branch   :", approval_branch.invoke({"exposure": 1800}), "/", approval_branch.invoke({"exposure": 100}))

# 3. Retries and fallbacks come free on any Runnable.
flaky = RunnableLambda(lambda x: (_ for _ in ()).throw(ValueError("primary down")))
print("fallback :", flaky.with_fallbacks([RunnableLambda(lambda x: "backup answer")]).invoke({}))

### Now break it

Four things the release desk needs. Try each one against a chain and watch what happens.

**One: route to different work depending on what the diagnosis found.**

`RunnableBranch` handles this, and for two or three branches it is fine. Once the branches have branches, you are writing a decision tree as nested lambdas inside a pipeline expression, and nobody can read it in a code review.

**Two: loop until the plan passes review.**

A chain is a straight line. There is no `|` that points backwards. You would write a Python `while` loop around the chain, which works, and now the interesting part of your control flow lives outside the thing you built the abstraction for.

**Three: stop and wait for a human to approve the fee waiver.**

This is the one that ends the argument. `chain.invoke()` is a function call. It returns or it raises. There is no third option where it hands you a token, goes to sleep, and picks up tomorrow when someone clicks approve. You can fake it by splitting into two chains and storing something in a database yourself, which means you are now writing a workflow engine.

**Four: survive a crash halfway through.**

If step four of six dies, a chain gives you nothing. Rerunning means redoing steps one to three, paying for those tokens again, and hoping step two was not a side effect.

Here is what people actually do when a chain stops fitting.

In [ ]:
# Runs offline. The hand-rolled orchestrator. Everyone writes this once.

def hand_rolled_release(container_id: str, party: str, approval_callback=None) -> dict:
    state = {"container_id": container_id, "party": party, "path": []}

    state["path"].append("intake")
    if len(container_id) != 11 or party not in {"maersk-ops", "cma-cgm-ops"}:
        state["path"].append("rejected")
        return state

    state["path"].append("diagnose")
    state["blockers"] = ["customs", "billing"]

    attempts = 0
    while True:                                    # the review loop
        attempts += 1
        state["path"].append("plan")
        state["exposure"] = 1800.0
        state["path"].append("review")
        if attempts >= 2:
            break
        if attempts > 10:                          # the cap you remember to add
            state["path"].append("gave_up")
            return state

    if state["exposure"] > 500:
        state["path"].append("approval")
        if approval_callback is None:              # the part that cannot pause
            state["path"].append("BLOCKED: no way to ask a human from inside this function")
            return state
        if not approval_callback(state):
            state["path"].append("rejected")
            return state

    state["path"].append("apply")
    return state


print("no approver:", hand_rolled_release("MSCU7391045", "maersk-ops")["path"])
print("with approver:", hand_rolled_release("MSCU7391045", "maersk-ops", lambda s: True)["path"])

That function is not wrong. It is what most teams ship, and it works right up until someone asks a reasonable question.

| Reasonable question | What you would have to build |
|---|---|
| "Can ops approve this tomorrow instead of now?" | Serialise the local variables, store them, rebuild them on resume |
| "The process died at step four, can we resume?" | Persist state after every step, with a resume entry point |
| "Which step is slow?" | Instrument every block by hand |
| "Can the four pre-arrival checks run at once?" | Restructure into async and merge the results yourself |
| "Show me the state as it was before approval" | Keep a history of every state version |
| "Why did it take the rejected path?" | Read the function and reconstruct the run in your head |

Every one of those is a checkpointing problem wearing a different hat. And that is the actual pitch for LangGraph: not that it makes agents smarter, but that it takes the state you were about to manage by hand and manages it properly.

## 3. Level two: your first StateGraph

Three ideas. Learn these and the rest is detail.

**Idea one: there is one state object, and it is a dict.**

You declare its shape with a `TypedDict`. Every node reads it and every node writes to it. No passing arguments between steps, no closures, no globals.

**Idea two: a node is a function that returns a partial update, not the whole state.**

The node gets the full state and returns only the keys it changed. LangGraph merges. If a node returns `{"exposure": 1800.0}`, everything else in the state stays exactly as it was.

**Idea three: a reducer decides how a new value combines with the old one.**

Default behaviour is overwrite. If two nodes both write `verdict`, the second wins. That is usually what you want for a scalar. For a list you almost never want it: you want the values to accumulate. You say so with `Annotated[list[str], operator.add]`, and now `+` is the merge rule for that key.

```mermaid
flowchart TD
    S["State dict, one per run"] --> N1["node A reads state"]
    N1 --> U1["returns partial update"]
    U1 --> R["reducer merges into state"]
    R --> N2["node B reads the merged state"]
    N2 --> U2["returns partial update"]
    U2 --> R2["reducer merges again"]
```

The reducer is the piece that surprises people, so here it is on its own before any of the release desk logic.

In [ ]:
# Runs offline. Reducers, in isolation.

class Demo(TypedDict):
    verdict: str                                   # no reducer: last write wins
    log: Annotated[list[str], operator.add]        # reducer: writes accumulate


def first(state: Demo) -> dict:
    return {"verdict": "clear", "log": ["first ran"]}


def second(state: Demo) -> dict:
    return {"verdict": "blocked", "log": ["second ran"]}


demo = StateGraph(Demo)
demo.add_node("first", first)
demo.add_node("second", second)
demo.add_edge(START, "first")
demo.add_edge("first", "second")
demo.add_edge("second", END)

result = demo.compile().invoke({"verdict": "", "log": []})
print("verdict (overwritten):", result["verdict"])
print("log     (accumulated):", result["log"])
print()
print("The two keys were written by the same two nodes. Only the reducer differs.")

### Now the release desk, as a graph

Same six stages, with the diagnosis stubbed for a moment so the shape stays visible. Two new things appear: `add_conditional_edges`, and a routing function that returns the name of the next node.

A routing function is ordinary Python. It reads state and returns a string. That matters more than it sounds: **your business rules live in a function you can unit test, not in a prompt you can only hope about.**

In [ ]:
# Runs offline. The control flow, with the model work stubbed.

APPROVAL_THRESHOLD_USD = 500.0
AUTHORIZED_PARTIES = {"maersk-ops", "cma-cgm-ops", "pierpoint-internal"}


class ReleaseState(TypedDict):
    container_id: str
    party: str
    problems: Annotated[list[str], operator.add]
    diagnosis: str
    exposure: float | None
    approved: bool
    path: Annotated[list[str], operator.add]


def intake(state: ReleaseState) -> dict:
    problems = []
    cid = state["container_id"].strip().upper()
    if not (len(cid) == 11 and cid[:4].isalpha() and cid[4:].isdigit()):
        problems.append("container_id is not ISO 6346 shape")
    if state["party"].strip().lower() not in AUTHORIZED_PARTIES:
        problems.append("party is not authorized to request a release")
    return {"path": ["intake"], "problems": problems}


def diagnose(state: ReleaseState) -> dict:
    return {"path": ["diagnose"], "diagnosis": "customs documentary hold plus 1800 USD demurrage"}


def plan(state: ReleaseState) -> dict:
    exposure = 1800.0 if "demurrage" in state["diagnosis"] else 0.0
    return {"path": ["plan"], "exposure": exposure}


def approval(state: ReleaseState) -> dict:
    return {"path": ["approval"], "approved": True}


def apply_effects(state: ReleaseState) -> dict:
    return {"path": ["apply_effects"]}


def rejected(state: ReleaseState) -> dict:
    return {"path": ["rejected"]}


def audit(state: ReleaseState) -> dict:
    return {"path": ["audit"]}


# --- routing functions: plain Python, unit testable, visible in a diff ---
def after_intake(state: ReleaseState) -> Literal["diagnose", "rejected"]:
    return "rejected" if state["problems"] else "diagnose"


def after_approval(state: ReleaseState) -> Literal["apply_effects", "rejected"]:
    return "apply_effects" if state["approved"] else "rejected"


builder = StateGraph(ReleaseState)
for name, fn in [("intake", intake), ("diagnose", diagnose), ("plan", plan), ("approval", approval),
                 ("apply_effects", apply_effects), ("rejected", rejected), ("audit", audit)]:
    builder.add_node(name, fn)

builder.add_edge(START, "intake")
builder.add_conditional_edges("intake", after_intake, ["diagnose", "rejected"])
builder.add_edge("diagnose", "plan")
builder.add_edge("plan", "approval")
builder.add_conditional_edges("approval", after_approval, ["apply_effects", "rejected"])
builder.add_edge("apply_effects", "audit")
builder.add_edge("rejected", "audit")
builder.add_edge("audit", END)

release_graph = builder.compile()

BLANK = {"container_id": "", "party": "", "problems": [], "diagnosis": "",
         "exposure": None, "approved": False, "path": []}

good = release_graph.invoke({**BLANK, "container_id": "MSCU7391045", "party": "maersk-ops"})
bad = release_graph.invoke({**BLANK, "container_id": "MSC7391045", "party": "unknown-broker"})

print("authorised request :", " -> ".join(good["path"]))
print("bad request        :", " -> ".join(bad["path"]))
print("why it was rejected:", bad["problems"])

In [ ]:
# LangGraph draws itself. Paste this into a markdown cell, or read it as is.
print(release_graph.get_graph().draw_mermaid())

```mermaid
graph TD;
    __start__([__start__]):::first
    intake(intake)
    diagnose(diagnose)
    plan(plan)
    approval(approval)
    apply_effects(apply_effects)
    rejected(rejected)
    audit(audit)
    __end__([__end__]):::last
    __start__ --> intake;
    intake -.-> diagnose;
    intake -.-> rejected;
    diagnose --> plan;
    plan --> approval;
    approval -.-> apply_effects;
    approval -.-> rejected;
    apply_effects --> audit;
    rejected --> audit;
    audit --> __end__;
    classDef first fill-opacity:0
    classDef last fill:#bfb6fc
```

Solid arrows are fixed edges. Dotted arrows are conditional ones, meaning a Python function chose that path at runtime.

This diagram is generated from the same object that executes. It cannot drift from the code, which is not true of any architecture diagram you have ever maintained in a wiki. When someone asks what the system does, you run one line and hand them the answer.

Two habits worth forming now, while the graph is still small:

- **Routing functions get a return type of `Literal[...]`.** Your editor then tells you when a node name is wrong, instead of LangGraph telling you at runtime.
- **Pass the destination list to `add_conditional_edges`.** It is optional, and without it the drawn diagram cannot show where the dotted arrows go.

## 4. What the graph gives you that the chain could not

The graph above still does not do anything the hand-rolled function could not. Time to collect on the promise.

Everything in this part comes from one decision: **compile with a checkpointer.** A checkpointer writes the state to storage after every step. Once state lives outside the Python call stack, five separate capabilities fall out of the same mechanism.

```mermaid
flowchart TD
    CP["Checkpointer: state saved after every step"] --> A["Pause and resume for human approval"]
    CP --> B["Survive a crash and continue"]
    CP --> C["Memory across separate invocations"]
    CP --> D["Time travel to an earlier state"]
    CP --> E["Inspect a paused run and see what is next"]
```

Along with the checkpointer comes the **thread**. Every run belongs to a thread id you supply in the config, and the thread is the unit of state. Same thread id means continue that conversation. New thread id means start fresh. In production this is your correlation id, so make it the same string you put in your logs.

In [ ]:
# Runs offline. The approval gate, done properly.

def approval_with_gate(state: ReleaseState) -> dict:
    exposure = state.get("exposure")

    # Fail closed: unknown exposure is treated exactly like too much exposure.
    # A missing number must never behave like a zero.
    if exposure is not None and exposure <= APPROVAL_THRESHOLD_USD:
        return {"path": ["approval"], "approved": True}

    decision = interrupt({
        "question": "Approve this fee waiver?",
        "container_id": state["container_id"],
        "exposure_usd": exposure,
        "exposure_known": exposure is not None,
        "threshold_usd": APPROVAL_THRESHOLD_USD,
    })

    return {"path": ["approval"], "approved": bool(decision.get("approved"))}


gated = StateGraph(ReleaseState)
for name, fn in [("intake", intake), ("diagnose", diagnose), ("plan", plan),
                 ("approval", approval_with_gate), ("apply_effects", apply_effects),
                 ("rejected", rejected), ("audit", audit)]:
    gated.add_node(name, fn)
gated.add_edge(START, "intake")
gated.add_conditional_edges("intake", after_intake, ["diagnose", "rejected"])
gated.add_edge("diagnose", "plan")
gated.add_edge("plan", "approval")
gated.add_conditional_edges("approval", after_approval, ["apply_effects", "rejected"])
gated.add_edge("apply_effects", "audit")
gated.add_edge("rejected", "audit")
gated.add_edge("audit", END)

# The one line that changes everything.
gated_graph = gated.compile(checkpointer=InMemorySaver())

thread = {"configurable": {"thread_id": "req-0001"}}
paused = gated_graph.invoke({**BLANK, "container_id": "MSCU7391045", "party": "maersk-ops"}, thread)

print("path so far    :", " -> ".join(paused["path"]))
print("did it finish? :", "apply_effects" in paused["path"])
print()
interrupt_obj = paused["__interrupt__"][0]
print("what it is asking:", json.dumps(interrupt_obj.value, indent=2))
print("resume token     :", interrupt_obj.id[:24], "...")
print("paused before    :", gated_graph.get_state(thread).next)

`invoke()` returned. The process is free. The state sits in the checkpointer with a note saying which node is next.

Somewhere else entirely, minutes or days later, a human approves. You resume by calling the same graph with the same thread id and a `Command(resume=...)`.

In [ ]:
# Runs offline. Resume with an approval.
resumed = gated_graph.invoke(
    Command(resume={"approved": True, "approver": "ops_lead_2", "note": "waiver agreed under dispute policy"}),
    thread,
)
print("APPROVED path:", " -> ".join(resumed["path"]))

# Same thing, denied, on a fresh thread.
deny_thread = {"configurable": {"thread_id": "req-0002"}}
gated_graph.invoke({**BLANK, "container_id": "MSCU7391045", "party": "maersk-ops"}, deny_thread)
denied = gated_graph.invoke(Command(resume={"approved": False, "approver": "ops_lead_2"}), deny_thread)
print("DENIED   path:", " -> ".join(denied["path"]))

Read those two paths carefully.

`intake` and `diagnose` and `plan` each appear **once**, even though the graph was invoked twice. Completed nodes are not replayed. The expensive diagnosis was not paid for a second time, because its result was already in the checkpoint.

Denial went to `rejected` and then `audit`. It did not raise. That is a design choice worth making explicit: **a denied approval is a destination, not an error.** If you raise instead, you lose the audit node, the customer notification, and the result object, and you gain a stack trace that tells you nothing an edge would not have told you.

### The footgun everybody hits once

Completed nodes do not replay. The **interrupted node itself** does replay, from its first line, when you resume.

Read that twice, because it means anything the node did before calling `interrupt()` happens again.

In [ ]:
# Runs offline. The double-charge, demonstrated.

SIDE_EFFECTS: list[str] = []


class Tiny(TypedDict):
    log: Annotated[list[str], operator.add]


def careless_node(state: Tiny) -> dict:
    SIDE_EFFECTS.append("charged the customer")        # BEFORE the interrupt
    decision = interrupt({"approve?": True})
    SIDE_EFFECTS.append(f"applied {decision}")         # AFTER the interrupt
    return {"log": ["done"]}


tiny = StateGraph(Tiny)
tiny.add_node("careless", careless_node)
tiny.add_edge(START, "careless")
tiny.add_edge("careless", END)
tiny_graph = tiny.compile(checkpointer=InMemorySaver())

t = {"configurable": {"thread_id": "footgun"}}
tiny_graph.invoke({"log": []}, t)
print("after pause :", SIDE_EFFECTS)

tiny_graph.invoke(Command(resume="yes"), t)
print("after resume:", SIDE_EFFECTS)
print()
print("customer charged twice:", SIDE_EFFECTS.count("charged the customer") == 2)

The customer was charged twice, and nothing raised.

The fix is a rule you can enforce in review, not a library feature.

| Rule | In practice |
|---|---|
| `interrupt()` goes at the **top** of its node | Nothing above it can run twice |
| A node that interrupts does nothing else | It asks, records the answer, returns. That is the whole node |
| Side effects live in their own node, downstream of the gate | `apply_effects` never interrupts, so it never replays |

Look back at `approval_with_gate`. It reads state, decides, interrupts, returns a verdict. It touches nothing. `apply_effects` sits on the other side of a conditional edge and is the only node allowed to write. That separation is not stylistic, it is the thing that makes the footgun impossible.

In [ ]:
# Runs offline. Three more controls that all come from the same checkpointer.

# 1. The cap. A cyclic graph without one is an unbounded bill.
class Spin(TypedDict):
    n: int


def spin(state: Spin) -> dict:
    return {"n": state["n"] + 1}


spinner = StateGraph(Spin)
spinner.add_node("spin", spin)
spinner.add_edge(START, "spin")
spinner.add_edge("spin", "spin")          # a deliberate loop with no exit
spin_graph = spinner.compile()

try:
    spin_graph.invoke({"n": 0}, {"recursion_limit": 8})
except GraphRecursionError as exc:
    print("1. cap hit  :", type(exc).__name__, "|", str(exc)[:60])

# 2. Per-node retry. The graph handles transient failure so your node body does not.
attempts = {"n": 0}


def flaky(state: Tiny) -> dict:
    attempts["n"] += 1
    if attempts["n"] < 3:
        raise ValueError("bedrock throttled")
    return {"log": ["succeeded"]}


retry_builder = StateGraph(Tiny)
retry_builder.add_node("flaky", flaky,
                       retry_policy=RetryPolicy(max_attempts=4, initial_interval=0.01,
                                                retry_on=ValueError))
retry_builder.add_edge(START, "flaky")
retry_builder.add_edge("flaky", END)
print("2. retry    :", retry_builder.compile().invoke({"log": []}), "after", attempts["n"], "attempts")

# 3. Time travel. Every checkpoint is still there.
history = list(gated_graph.get_state_history(thread))
print("3. history  :", len(history), "checkpoints on thread req-0001")
for snap in reversed(history[:4]):
    print(f"     next={str(snap.next):<18} path={snap.values.get('path')}")

`recursion_limit` is a per-invocation setting, not a graph setting, and its default is 25. That default is the reason a runaway loop usually surfaces as a confusing error rather than a runaway bill. Set it deliberately anyway: the number should be a decision, not an accident.

`RetryPolicy` has a default `retry_on` aimed at connection and API errors. It will not retry your `ValueError` unless you say so, which the cell above does explicitly.

`get_state_history` is genuinely useful in an incident. You get every checkpoint, newest first, each with the state as it was and the node that was about to run. You can also pass a specific checkpoint id back into `invoke` to fork the run from that point, which is how you answer "what would have happened if ops had said no".

## 5. Parallel work

Four pre-arrival checks: customs, cranes, weather, documents. None of them needs the others. Running them one after another is four times the wall clock for no reason.

In a graph you express this by drawing four edges out of one node. LangGraph runs everything it can in a step, in parallel, and this is exactly where reducers stop being a curiosity and start being load bearing. Four nodes are about to write to the same key in the same step. Without a reducer that is a race. With `operator.add` it is a merge.

In [ ]:
# Runs offline. Static fan-out, four branches, one reducer.

class Sweep(TypedDict):
    checks: Annotated[list[str], operator.add]
    verdict: str


VESSEL_FILE = {
    "customs": "3 flagged, 1 unresolved",
    "cranes": "2 of 3 available",
    "weather": "gusts 28 knots, limit is 32",
    "documents": "DG declaration missing",
}


def make_check(area: str):
    def run(state: Sweep) -> dict:
        time.sleep(0.2)                                    # stand-in for a model call
        note = VESSEL_FILE[area]
        blocked = "unresolved" in note or "missing" in note
        return {"checks": [f"{area}:{'BLOCKED' if blocked else 'READY'}"]}
    return run


def reduce_verdict(state: Sweep) -> dict:
    blocked = [c.split(":")[0] for c in state["checks"] if c.endswith("BLOCKED")]
    return {"verdict": f"BLOCKED by {', '.join(sorted(blocked))}" if blocked else "READY"}


sweep = StateGraph(Sweep)
for area in VESSEL_FILE:
    sweep.add_node(area, make_check(area))
    sweep.add_edge(START, area)                            # fan out
    sweep.add_edge(area, "merge")                          # fan in
sweep.add_node("merge", reduce_verdict)
sweep.add_edge("merge", END)

started = time.time()
out = sweep.compile().invoke({"checks": [], "verdict": ""})
print("checks  :", sorted(out["checks"]))
print("verdict :", out["verdict"])
print("elapsed :", round(time.time() - started, 2), "s   (serial would be 0.8s)")

0.2 seconds, not 0.8. Four model calls became one call's worth of waiting.

Note what the merge node is: plain Python, no model. The rule for combining four verdicts is written down, so a model has no business enforcing it. If you find yourself asking an LLM to summarise four READY/BLOCKED strings into one verdict, you have added latency, cost and a failure mode in exchange for nothing.

Two more things to know about parallel branches:

- **Token cost is identical.** Parallelism buys wall clock only. If the answer is "we need this cheaper", parallelism is the wrong lever.
- **The weather branch is a trap.** 28 knots against a 32 knot limit is READY. If your branch says BLOCKED, that is a prompt problem, not a topology problem, and no amount of graph structure fixes it.

Sometimes you do not know how many branches there are until the request arrives. `Send` covers that: a routing function returns a list of `Send` objects, each carrying its own payload, and each becomes its own parallel task.

In [ ]:
# Runs offline. Fan-out decided at runtime with Send.

class Fan(TypedDict):
    areas: list[str]
    results: Annotated[list[str], operator.add]


def dispatch(state: Fan):
    # One Send per area. The list length is not known until this runs.
    return [Send("worker", {"area": area}) for area in state["areas"]]


def worker(payload: dict) -> dict:
    # A Send'd node receives the payload, not the whole graph state.
    return {"results": [f"{payload['area']} checked"]}


fan = StateGraph(Fan)
fan.add_node("worker", worker)
fan.add_conditional_edges(START, dispatch, ["worker"])
fan.add_edge("worker", END)
fan_graph = fan.compile()

print("two areas  :", fan_graph.invoke({"areas": ["customs", "billing"], "results": []})["results"])
print("four areas :", fan_graph.invoke({"areas": ["customs", "billing", "damage", "equipment"],
                                        "results": []})["results"])

## 6. LangGraph wears two hats

This trips people up, so it is worth stating plainly. LangGraph shows up in two different roles, and the code looks similar in both.

**Hat one: LangGraph as the runtime under an agent.**

When you call `create_agent(model, tools)`, you get back a compiled LangGraph. The model-calls-tool-calls-model loop is a graph with a cycle in it. You did not draw that graph and you mostly do not need to think about it, but it is why an agent gets checkpointing, streaming and interrupts for free. The graph is the engine.

**Hat two: LangGraph as an explicit state machine you draw yourself.**

That is everything in Parts 3 to 5. You name the nodes, you write the routing functions, you own the topology. The graph is the blueprint.

```mermaid
flowchart TD
    subgraph HAT1 ["Hat one: runtime"]
      A["create_agent(model, tools)"] --> B["a cycle you did not draw"]
      B --> C["model, tools, model, tools, stop"]
    end
    subgraph HAT2 ["Hat two: state machine"]
      D["StateGraph(YourState)"] --> E["nodes and edges you did draw"]
      E --> F["your business rules, in Python"]
    end
```

The practical question is which hat fits which stage of your workflow.

| Situation | Hat | Why |
|---|---|---|
| "Answer questions using these four lookups" | Runtime | The tool loop is the whole job. Do not hand-build it |
| "Validate, plan, approve, apply, audit, in that order" | State machine | The order is the requirement |
| "Investigate until you find the blocker" | Runtime, wrapped in a cap | Loop length is unknown, so let the agent loop |
| "Never release without an approver" | State machine | An invariant belongs on an edge, not in a prompt |

And the answer for a real system is usually both, which is what Part 9 builds.

### When NOT to reach for LangGraph

Being straight about this earns you credibility when you do recommend it.

| Situation | Use instead |
|---|---|
| A prompt and a parser | An LCEL chain |
| One agent with a handful of tools | `create_agent`, without drawing a graph |
| A fixed sequence with no branching, no pausing, no retry | A chain, or plain Python |
| A pure data pipeline with no model in it | Airflow, Step Functions, dbt, whatever you already run |

LangGraph earns its complexity when you need **branching, cycles, pausing, or durable state**. If none of those four words describes your problem, you are paying for a framework you will not use.

## 7. Swarms

Back to stage two, the one row in the table that said no. Why is the container stuck? You cannot draw that flow because the flow depends on what each check turns up.

A swarm is a set of peer agents that pass the case to each other. Nobody is in charge. Each specialist checks their own area and, if the trail leads elsewhere, hands off.

The mechanism in LangGraph is one idea: **a tool that returns a `Command` instead of a string.**

Normally a tool returns text and the agent keeps going. If it returns a `Command`, that Command is an instruction to the graph itself. `Command(goto="billing")` means route to the billing node. `graph=Command.PARENT` means the instruction applies to the graph one level up, which is how a tool inside one agent moves control to a different agent.

```mermaid
flowchart TD
    A["customs agent"] --> T["calls transfer_to_billing"]
    T --> C["tool returns Command(goto='billing', graph=PARENT)"]
    C --> P["parent graph routes to the billing node"]
    P --> B["billing agent continues, same message history"]
```

We will build this three ways: by hand, with the prebuilt library, and as a supervisor. Same problem each time, so the differences are the lesson.

First, the specialists and their read-only tools. Read-only is invariant I3, and it is enforced by what these tools *are*, not by what the prompt asks for. There is no write tool in this list, so no specialist can write.

In [ ]:
# Runs offline. Four read-only lookups.

CUSTOMS = {"MSCU7391045": {"hold": "documentary", "reason": "invoice value mismatch", "cleared": False},
           "CAIU9083321": {"hold": "none", "reason": "", "cleared": True}}
BILLING = {"MSCU7391045": {"demurrage_days": 6, "accrued_usd": 1800.0, "dispute_open": True},
           "CAIU9083321": {"demurrage_days": 0, "accrued_usd": 0.0, "dispute_open": False}}
SURVEY = {"MSCU7391045": {"survey_done": False, "damage": "none reported"},
          "CAIU9083321": {"survey_done": True, "damage": "none"}}
EQUIPMENT = {"MSCU7391045": {"fault": "none", "reefer_required": False},
             "CAIU9083321": {"fault": "reefer plug bay R04 no power", "reefer_required": True}}

TOOL_CALLS: list[str] = []


def _lookup(table: dict, container_id: str, label: str) -> str:
    """Shared not-found contract. Never raise, never return an empty string: the model
    needs a true sentence it can relay instead of a gap it will fill in for itself."""
    TOOL_CALLS.append(label)
    record = table.get(container_id.strip().upper())
    if record is None:
        return f"No {label} record exists for container '{container_id}'."
    return json.dumps(record)


@tool
def customs_status(container_id: str) -> str:
    """Read the customs hold status for one container. Argument is an ISO 6346 number."""
    return _lookup(CUSTOMS, container_id, "customs")


@tool
def billing_status(container_id: str) -> str:
    """Read demurrage, accrued charges and dispute state for one container."""
    return _lookup(BILLING, container_id, "billing")


@tool
def damage_survey(container_id: str) -> str:
    """Read the damage survey record for one container."""
    return _lookup(SURVEY, container_id, "survey")


@tool
def equipment_status(container_id: str) -> str:
    """Read equipment and reefer power faults affecting one container."""
    return _lookup(EQUIPMENT, container_id, "equipment")


READ_ONLY = {"customs": customs_status, "billing": billing_status,
             "damage": damage_survey, "equipment": equipment_status}
print("read-only tools:", [t.name for t in READ_ONLY.values()])
print("write tools    : none, by construction. That is invariant I3.")

### Way one: build the handoff yourself

Worth doing once even though a library exists, because the library hides the single most important detail.

The handoff tool needs three things: a target, a way to reach the graph above it, and the message history. That last one is where people lose data, so the cell below shows the wrong version and the right version side by side.

In [ ]:
# Runs offline. The handoff tool, wrong and right.

def make_handoff_naive(target: str):
    """WRONG. Only what is inside `update` reaches the parent graph."""
    @tool(f"transfer_to_{target}_naive", description=f"Hand the case to {target}.")
    def handoff(reason: str, tool_call_id: Annotated[str, InjectedToolCallId]) -> Command:
        note = ToolMessage(f"handed to {target}: {reason}",
                           name=f"transfer_to_{target}_naive", tool_call_id=tool_call_id)
        return Command(goto=target, graph=Command.PARENT,
                       update={"messages": [note], "active": target})
    return handoff


def make_handoff(target: str):
    """RIGHT. InjectedState hands the tool the inner agent's own message list, and we
    pass all of it upward. The add_messages reducer dedupes by message id, so nothing
    is doubled. Without this, everything the first specialist discovered is discarded."""
    @tool(f"transfer_to_{target}", description=f"Hand the case to the {target} specialist.")
    def handoff(reason: str,
                state: Annotated[Any, InjectedState],
                tool_call_id: Annotated[str, InjectedToolCallId]) -> Command:
        note = ToolMessage(f"handed to {target}: {reason}",
                           name=f"transfer_to_{target}", tool_call_id=tool_call_id)
        return Command(goto=target, graph=Command.PARENT,
                       update={"messages": [*state["messages"], note], "active": target})
    return handoff


print("naive tool:", make_handoff_naive("billing").name)
print("real tool :", make_handoff("billing").name)
print()
print("The difference is one argument. It decides whether the parent graph sees the")
print("first specialist's tool calls or only the sentence saying it gave up.")

In [ ]:
# [needs bedrock] The hand-rolled swarm.

class SwarmState(MessagesState):
    """MessagesState already carries `messages` with the add_messages reducer.
    We add one field so the graph knows who currently holds the case."""
    active: str


SPECIALIST_BRIEF = (
    "You are the {area} specialist on the PierPoint release desk.\n"
    "Check your own area with your tool first, then report what you found in two sentences.\n"
    "If your finding points at another area, call the matching transfer tool with a specific reason.\n"
    "If nothing else needs checking, state the blocking reason and stop.\n"
    "Do not hand back to a specialist who has already reported. You investigate only, "
    "you cannot change anything."
)

AREAS = ["customs", "billing", "damage", "equipment"]


def build_specialist(area: str):
    others = [make_handoff(other) for other in AREAS if other != area]
    return create_agent(
        model=chat(REASONING_MODEL, temperature=0.2, max_tokens=700),
        tools=[READ_ONLY[area], *others],
        system_prompt=SPECIALIST_BRIEF.format(area=area),
        name=area,
    )


def build_manual_swarm():
    builder = StateGraph(SwarmState)
    for area in AREAS:
        # The compiled agent goes in as a node. It speaks MessagesState, so it shares
        # the parent's message list. `destinations` is what lets the diagram draw the
        # handoff arrows, which are otherwise invisible to the renderer.
        builder.add_node(area, build_specialist(area),
                         destinations=tuple(a for a in AREAS if a != area) + (END,))
    builder.add_edge(START, "customs")
    return builder.compile()


manual_swarm = build_manual_swarm()

# Every dotted arrow below is a possible handoff, not a step that will happen.
# The actual route is chosen by the models at runtime, which is the whole point.
for line in manual_swarm.get_graph().draw_mermaid().splitlines():
    if "-.->" in line or "-->" in line:
        print(line.strip())

```mermaid
flowchart TD
    START(["start"]) --> C["customs"]
    C -.-> B["billing"]
    C -.-> D["damage"]
    C -.-> E["equipment"]
    B -.-> C
    B -.-> D
    B -.-> E
    D -.-> C
    D -.-> B
    D -.-> E
    E -.-> C
    E -.-> B
    E -.-> D
    C -.-> F(["END"])
    B -.-> F
    D -.-> F
    E -.-> F
```

Twelve possible handoffs between four agents, and every one of them was written by a single tool factory. Encoding the same reachability as fixed edges in a control graph would mean twelve edges plus the conditions to guard them, and a thirteenth the day someone adds a fifth blocker type.

In [ ]:
# [needs bedrock] Run it on two containers with different blockers.

def show_swarm_run(result: dict, label: str) -> None:
    print(f"\n=== {label} ===")
    hops, seen = [], set()
    for m in result["messages"]:
        who = getattr(m, "name", None)
        if m.__class__.__name__ == "AIMessage" and who and who not in seen:
            seen.add(who)
        if m.__class__.__name__ == "AIMessage" and who:
            hops.append(who)
    ordered, last = [], None
    for h in hops:
        if h != last:
            ordered.append(h)
            last = h
    print("path        :", " -> ".join(ordered))
    print("tool calls  :", TOOL_CALLS)
    print("final answer:", result["messages"][-1].content[:220])


if BEDROCK_OK:
    for cid in ("MSCU7391045", "CAIU9083321"):
        TOOL_CALLS.clear()
        out = build_manual_swarm().invoke(
            {"messages": [{"role": "user", "content": f"Why has container {cid} not been released?"}],
             "active": "customs"},
            {"recursion_limit": 20},
        )
        show_swarm_run(out, cid)
else:
    print("skipped, no bedrock")

Two containers, four agents, and the paths differ. That is the whole point, and it is worth saying out loud what just happened: **the route through those agents is not in your source code.** You wrote four nodes and a handoff tool. The sequence came from the model at runtime.

That is the swarm's only real advantage, and it is a big one for this stage. Adding a fifth blocker type means adding a fifth node. Encoding the same thing as a graph means enumerating the combinations, and four blockers that can co-occur in pairs is ten paths before you have handled anything unusual.

Now the cost side, which is equally real:

- `recursion_limit` is the only thing standing between you and a loop. Set it deliberately.
- The path is in the message history, not in a structured field. You know who spoke. You do not know which sentence came from which tool result.
- Two polite agents can hand back and forth. A prompt line telling them not to helps, and is not a guarantee.

### Way two: the prebuilt library

`langgraph-swarm` packages exactly what you just built: a handoff tool factory, an `active_agent` field, and a router that resumes the last active agent when a new message arrives on the same thread.

Use it when you want the standard behaviour. Build it yourself when you need a custom state field, a different handoff payload, or a fence around who may hand off to whom.

In [ ]:
# [needs bedrock] Same swarm, using the library.
from langgraph_swarm import create_handoff_tool, create_swarm


def build_prebuilt_swarm():
    agents = []
    for area in AREAS:
        handoffs = [create_handoff_tool(agent_name=other,
                                        description=f"Ask the {other} specialist for help.")
                    for other in AREAS if other != area]
        agents.append(create_agent(
            model=chat(REASONING_MODEL, temperature=0.2, max_tokens=700),
            tools=[READ_ONLY[area], *handoffs],
            system_prompt=SPECIALIST_BRIEF.format(area=area),
            name=area,
        ))
    # A checkpointer here is what makes the swarm resumable: send a second message on
    # the same thread and it lands with whichever agent was last active.
    return create_swarm(agents, default_active_agent="customs").compile(checkpointer=InMemorySaver())


sample_tool = create_handoff_tool(agent_name="billing")
print("generated tool name:", sample_tool.name)

if BEDROCK_OK:
    swarm = build_prebuilt_swarm()
    TOOL_CALLS.clear()
    out = swarm.invoke(
        {"messages": [{"role": "user", "content": "Why has container MSCU7391045 not been released?"}]},
        {"configurable": {"thread_id": "swarm-1"}, "recursion_limit": 20},
    )
    print("active_agent at end:", out.get("active_agent"))
    print("tool calls         :", TOOL_CALLS)
    print("answer             :", out["messages"][-1].content[:200])
else:
    print("skipped, no bedrock")

### Way three: a supervisor

Peers handing to each other is one topology. Another is a coordinator that decides who goes next and gets the case back each time.

Same agents, different wiring, and different properties. Everything flows through one node, which means one place to log, one place to enforce a policy, and one bottleneck.

```mermaid
flowchart TD
    S["supervisor"] -.-> C["customs"]
    S -.-> B["billing"]
    S -.-> D["damage"]
    S -.-> E["equipment"]
    C --> S
    B --> S
    D --> S
    E --> S
    S -.-> X["END"]
```

The version below uses a plain Python supervisor so the topology stays visible without a model call. In production the supervisor is usually an agent that picks the next specialist, and swapping one for the other changes nothing about the wiring.

In [ ]:
# Runs offline. Supervisor topology with a deterministic coordinator.

class SupState(TypedDict):
    container_id: str
    asked: Annotated[list[str], operator.add]
    findings: Annotated[list[str], operator.add]


PLANNED = ["customs", "billing"]


def supervisor(state: SupState) -> dict:
    remaining = [a for a in PLANNED if a not in state["asked"]]
    return {"asked": [remaining[0]] if remaining else []}


def route_from_supervisor(state: SupState) -> Literal["customs", "billing", "__end__"]:
    if not state["asked"] or len(state["findings"]) >= len(state["asked"]):
        return END
    return state["asked"][-1]


def make_specialist_node(area: str):
    def run(state: SupState) -> dict:
        return {"findings": [f"{area}: checked {state['container_id']}"]}
    return run


sup = StateGraph(SupState)
sup.add_node("supervisor", supervisor)
for area in PLANNED:
    sup.add_node(area, make_specialist_node(area))
    sup.add_edge(area, "supervisor")
sup.add_edge(START, "supervisor")
sup.add_conditional_edges("supervisor", route_from_supervisor, PLANNED + [END])

out = sup.compile().invoke({"container_id": "MSCU7391045", "asked": [], "findings": []},
                           {"recursion_limit": 20})
print("asked   :", out["asked"])
print("findings:", out["findings"])

### The three, compared

| | Hand-rolled handoff | `langgraph-swarm` | Supervisor |
|---|---|---|---|
| Who picks next | The specialist holding the case | Same | The coordinator |
| Lines of your code | Most | Fewest | Middling |
| One place to log or gate | No | No | Yes, the supervisor |
| Custom state and payloads | Yes | Limited | Yes |
| Extra model call per hop | No | No | Yes, the coordinator thinks too |
| Fails by | Ping-pong, weak provenance | Same | Coordinator becomes a bottleneck and a single point of confusion |
| Reach for it when | You need a custom fence or payload | You want standard behaviour today | You need one throat to choke for routing |

None of these three is the "advanced" one. They are three shapes for three situations, and the honest default for a first build is the prebuilt library until you find a reason to leave it.

## 8. Graph or swarm

Here is the whole distinction in one line, and it is the line to carry out of this notebook.

> **The graph owns business invariants, approval gates, sequencing and side effects. The swarm owns bounded exploration where you cannot predict which specialist is needed next.**

Underneath that sentence sit two rules you can actually apply while reading someone's pull request.

**Rule one: if you can name the next step at design time, an LLM must not be the thing that picks it.**

**Rule two: if a step writes, it does not get to decide whether it runs.**

Both are testable. Rule one means counting the transitions a model chooses. Rule two means tracing every write to the gate above it.

```mermaid
flowchart TD
    S["A stage in your workflow"] --> W{"Does it write, move money, or reach a customer?"}
    W -->|Yes| G1["Graph node, below a gate, idempotent"]
    W -->|No| E{"Can you name the next step at design time?"}
    E -->|Yes| G2["Graph node or routing function"]
    E -->|No| H{"Is it read only, with a bounded hop count?"}
    H -->|Yes| SW["Swarm, capped, typed output at the boundary"]
    H -->|No| STOP["Not an agent problem yet. Narrow the scope first"]
```

| Concern | Graph | Swarm |
|---|---|---|
| Business invariants | Owns them. Routing functions are Python | Cannot enforce. A system prompt is a request |
| Sequencing | Owns it. The topology is the sequence | Emergent, and it changes run to run |
| Approval gates | Owns them. `interrupt()` in a gate node | No gate primitive |
| Side effects | Owns them. One node, below the gate | Must never write |
| Provenance | Owns it. `path`, per-node state, checkpoints | Message history says who spoke, not which claim came from where |
| Unknown next specialist | Needs combinatorial edges | Owns it. Handoff decides at runtime |
| Cost predictability | High, bounded by topology | Bounded only by the caps you set |

### The four ways teams get this wrong

| Anti-pattern | What breaks |
|---|---|
| Giving a swarm a write tool | Two specialists both issue the release. No gate, no idempotency, no audit |
| Encoding diagnosis as fourteen conditional edges | Every new blocker type is a code change and a deploy |
| Passing prose across the boundary into the control graph | Wording drifts, the parse breaks quietly, in production |
| Putting the approval rule in a system prompt | The model complies most of the time, and most is not a control |

## 9. The hybrid

Now put them together, which is the actual answer for the release desk and for most enterprise workflows.

The swarm becomes one node inside the control graph. Around it sits everything the swarm must not be trusted with.

```mermaid
flowchart TD
    START(["request"]) --> IN["intake, python"]
    IN -->|"problems"| RJ["rejected, python"]
    IN -->|"clean"| DG["diagnose, SWARM subgraph"]
    DG --> CT["contract, prose in types out"]
    CT --> PL["plan, model, typed output"]
    PL --> AP["approval, gate node, interrupt"]
    AP -->|"approved"| AE["apply_effects, python, idempotent"]
    AP -->|"denied"| RJ
    AE --> AU["audit, python"]
    RJ --> AU
    AU --> FIN(["END"])
```

There is one node here you have not seen yet, and it is the most important one in the picture.

**The contract node.** The swarm produces paragraphs. Paragraphs are fine for a human reading a case file. They are not an input to a system that moves money. The contract node takes the swarm's prose and returns a validated object, and from that point on the rest of the graph reads fields instead of guessing at sentences.

Three reasons it is not optional:

1. A parse of prose fails silently. A validation error fails loudly, in the run that caused it.
2. The typed object is what you store for audit. A paragraph is a recollection, not evidence.
3. It is the seam where you could replace the swarm with a single agent, or with a rules engine, and never touch the control graph.

One extra cheap model call per request. Cheaper than one wrong release.

In [ ]:
# Runs offline. The contracts, built so they cannot raise.

def _null_to_list(value: Any) -> Any:
    """Models emit null for an empty list. Pydantic's default_factory only fires when the
    key is ABSENT, so an explicit null still raises. Normalise it here."""
    return [] if value is None else value


def _to_money(value: Any) -> Any:
    """Return None for anything unreadable, and never invent a zero. None is what makes
    the approval gate fail closed, and a missing number is the classic way an
    unapproved waiver gets issued with nothing throwing."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value) if value >= 0 else None
    if isinstance(value, str):
        import re
        match = re.search("-?[0-9]+(?:[.][0-9]+)?", value.replace(",", ""))
        if match:
            parsed = float(match.group())
            return parsed if parsed >= 0 else None
    return None


def _to_bool(value: Any) -> Any:
    """Unreadable means blocking. Wrong in the safe direction costs one extra step.
    Wrong in the other direction releases a container that should not move."""
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        text = value.strip().lower()
        if text in {"true", "yes", "y", "1", "blocking"}:
            return True
        if text in {"false", "no", "n", "0", "clear"}:
            return False
    return True


from pydantic import BeforeValidator


class Finding(BaseModel):
    area: Literal["customs", "billing", "damage", "equipment"] = "customs"
    blocking: Annotated[bool, BeforeValidator(_to_bool)] = True
    detail: str = Field(default="", max_length=300)


class DiagnosisReport(BaseModel):
    container_id: str = ""
    findings: Annotated[list[Finding], BeforeValidator(_null_to_list)] = Field(default_factory=list)
    primary_blocker: Literal["customs", "billing", "damage", "equipment", "none"] = "none"
    unresolved: Annotated[list[str], BeforeValidator(_null_to_list)] = Field(default_factory=list)


class PlanStep(BaseModel):
    action: Literal["clear_customs_hold", "waive_demurrage", "post_charge",
                    "dispatch_equipment_repair", "request_survey", "notify_party"] = "notify_party"
    target: str = ""
    rationale: str = Field(default="", max_length=200)


class RemediationPlan(BaseModel):
    container_id: str = ""
    steps: Annotated[list[PlanStep], BeforeValidator(_null_to_list)] = Field(default_factory=list)
    financial_exposure_usd: Annotated[float | None, BeforeValidator(_to_money)] = None
    customer_message: str = Field(default="", max_length=600)


# Prove the awkward cases before any of it touches a model.
print("explicit null      :", DiagnosisReport(**{"unresolved": None}).unresolved)
print("money as prose     :", RemediationPlan(**{"financial_exposure_usd": "about 1,800 USD"}).financial_exposure_usd)
print("money missing      :", RemediationPlan().financial_exposure_usd, "  <- unknown, not zero")
print("unreadable boolean :", Finding(**{"blocking": "not sure"}).blocking, " <- fails closed")

In [ ]:
# Runs offline. Node bodies for the control graph.

EFFECT_LEDGER: dict[str, dict] = {}


class DeskState(TypedDict):
    correlation_id: str
    container_id: str
    party: str
    problems: Annotated[list[str], operator.add]
    transcript: str
    report: dict
    plan: dict
    approved: bool
    approver: str
    rejection_reason: str
    path: Annotated[list[str], operator.add]


def desk_intake(state: DeskState) -> dict:
    problems = []
    cid = state["container_id"].strip().upper()
    if not (len(cid) == 11 and cid[:4].isalpha() and cid[4:].isdigit()):
        problems.append("container_id is not ISO 6346 shape")
    if state["party"].strip().lower() not in AUTHORIZED_PARTIES:
        problems.append("party is not authorized to request a release")
    return {"path": ["intake"], "problems": problems, "container_id": cid}


def approval_required(exposure_usd: float | None, threshold_usd: float) -> bool:
    """The policy, as a plain function.

    Pulled out of the node on purpose. `interrupt()` only works inside a running graph,
    so a rule left inside the node body can only be tested by running the whole graph.
    Out here it is four lines and four assertions."""
    return exposure_usd is None or exposure_usd > threshold_usd


def desk_approval(state: DeskState) -> dict:
    """The gate. It reads, it asks, it records. It touches nothing else, which is why
    replaying it on resume is harmless."""
    exposure = _to_money(state["plan"].get("financial_exposure_usd"))

    if not approval_required(exposure, APPROVAL_THRESHOLD_USD):
        return {"path": ["approval"], "approved": True, "approver": "policy"}

    decision = interrupt({
        "question": "Approve this remediation plan?",
        "container_id": state["container_id"],
        "exposure_usd": exposure,
        "exposure_known": exposure is not None,
        "threshold_usd": APPROVAL_THRESHOLD_USD,
        "steps": [s.get("action") for s in state["plan"].get("steps") or []],
    })

    approved = bool(decision.get("approved"))
    return {"path": ["approval"], "approved": approved,
            "approver": decision.get("approver", "unknown"),
            "rejection_reason": "" if approved else decision.get("note", "approval denied")}


def desk_apply(state: DeskState) -> dict:
    """Invariant I1. The key makes a retry a no-op instead of a second release.
    In production this is a conditional write against a real table, not a dict."""
    key = f"{state['correlation_id']}:release:{state['container_id']}"
    if key in EFFECT_LEDGER:
        return {"path": ["apply_effects"]}
    effects = [{"action": s.get("action"), "target": s.get("target")}
               for s in state["plan"].get("steps") or []]
    effects.append({"action": "release_container", "target": state["container_id"]})
    EFFECT_LEDGER[key] = {"effects": effects, "approver": state["approver"]}
    return {"path": ["apply_effects"]}


def desk_rejected(state: DeskState) -> dict:
    reason = state.get("rejection_reason") or "; ".join(state["problems"]) or "not approved"
    return {"path": ["rejected"], "rejection_reason": reason}


def desk_audit(state: DeskState) -> dict:
    return {"path": ["audit"]}


def desk_after_intake(state: DeskState) -> Literal["diagnose", "rejected"]:
    return "rejected" if state["problems"] else "diagnose"


def desk_after_approval(state: DeskState) -> Literal["apply_effects", "rejected"]:
    return "apply_effects" if state["approved"] else "rejected"


print("control-graph nodes ready")

In [ ]:
# [needs bedrock] The two model nodes, and the adapter that wraps the swarm.

CONTRACT_BRIEF = (
    "Convert a release-desk investigation transcript into the required structure. "
    "Include one finding per area that was actually checked, and set blocking to true only "
    "where the transcript says the container cannot move. Never invent an area. "
    "Use an empty list where a list is empty, never null."
)

PLANNER_BRIEF = (
    "You are the PierPoint release desk planner. You receive a validated diagnosis as JSON.\n"
    "One step per blocking finding, no steps for areas that are clear. "
    "financial_exposure_usd is the total value PierPoint gives up or defers, and it must "
    "always be a number, using 0 when nothing is given up. "
    "Never admit liability in customer_message. Under 120 words."
)


def desk_diagnose(state: DeskState) -> dict:
    """The adapter. Control state goes in, swarm state comes out, prose comes back.

    The swarm has its own state schema, so something has to translate. Doing that
    translation in one named node is what keeps the swarm swappable."""
    swarm = build_manual_swarm()
    result = swarm.invoke(
        {"messages": [{"role": "user",
                       "content": f"Why has container {state['container_id']} not been released? "
                                  "Find every blocker."}],
         "active": "customs"},
        {"recursion_limit": 20},
    )
    parts = [f"[{m.name}] {m.content}" for m in result["messages"]
             if m.__class__.__name__ == "AIMessage" and getattr(m, "name", None) and m.content]
    return {"path": ["diagnose"], "transcript": "\n\n".join(parts)}


def desk_contract(state: DeskState) -> dict:
    model = chat(CHEAP_MODEL, temperature=0.0).with_structured_output(DiagnosisReport)
    prompt = ChatPromptTemplate.from_messages([("system", CONTRACT_BRIEF),
                                               ("human", "Container: {cid}\n\nTranscript:\n{t}")])
    report = (prompt | model).invoke({"cid": state["container_id"], "t": state["transcript"]})
    return {"path": ["contract"], "report": report.model_dump()}


def desk_plan(state: DeskState) -> dict:
    model = chat(REASONING_MODEL, temperature=0.2).with_structured_output(RemediationPlan)
    prompt = ChatPromptTemplate.from_messages([("system", PLANNER_BRIEF),
                                               ("human", "Diagnosis:\n{r}")])
    plan = (prompt | model).invoke({"r": json.dumps(state["report"])})
    return {"path": ["plan"], "plan": plan.model_dump()}


print("model nodes ready")

In [ ]:
# [needs bedrock] Assemble and draw the full system.

def build_release_desk():
    builder = StateGraph(DeskState)
    builder.add_node("intake", desk_intake)
    builder.add_node("diagnose", desk_diagnose)
    builder.add_node("contract", desk_contract)
    builder.add_node("plan", desk_plan)
    builder.add_node("approval", desk_approval)
    builder.add_node("apply_effects", desk_apply)
    builder.add_node("rejected", desk_rejected)
    builder.add_node("audit", desk_audit)

    builder.add_edge(START, "intake")
    builder.add_conditional_edges("intake", desk_after_intake, ["diagnose", "rejected"])
    builder.add_edge("diagnose", "contract")
    builder.add_edge("contract", "plan")
    builder.add_edge("plan", "approval")
    builder.add_conditional_edges("approval", desk_after_approval, ["apply_effects", "rejected"])
    builder.add_edge("apply_effects", "audit")
    builder.add_edge("rejected", "audit")
    builder.add_edge("audit", END)

    # One checkpointer, shared, so a thread paused for approval can be resumed later.
    return builder.compile(checkpointer=DESK_SAVER)


DESK_SAVER = InMemorySaver()
release_desk = build_release_desk()

for line in release_desk.get_graph().draw_mermaid().splitlines():
    if "-.->" in line or "-->" in line:
        print(line.strip())

DESK_BLANK = {"correlation_id": "", "container_id": "", "party": "", "problems": [],
              "transcript": "", "report": {}, "plan": {}, "approved": False,
              "approver": "", "rejection_reason": "", "path": []}

In [ ]:
# [needs bedrock] Run 1. Equipment fault, no money at stake, so policy auto-approves.

def run_desk(container_id: str, party: str, correlation_id: str | None = None):
    correlation_id = correlation_id or f"req-{uuid.uuid4().hex[:8]}"
    cfg = {"configurable": {"thread_id": correlation_id}, "recursion_limit": 40}
    state = {**DESK_BLANK, "correlation_id": correlation_id,
             "container_id": container_id, "party": party}
    return correlation_id, cfg, release_desk.invoke(state, cfg)


def show_desk(out: dict, cfg: dict) -> None:
    print("path      :", " -> ".join(out["path"]))
    if "__interrupt__" in out:
        print("PAUSED at :", release_desk.get_state(cfg).next)
        print("asking    :", json.dumps(out["__interrupt__"][0].value, indent=2)[:420])
        return
    if out.get("report"):
        print("blocker   :", out["report"].get("primary_blocker"))
    if out.get("plan"):
        print("exposure  :", out["plan"].get("financial_exposure_usd"))
        print("steps     :", [s.get("action") for s in out["plan"].get("steps") or []])
    if out.get("rejection_reason"):
        print("rejected  :", out["rejection_reason"])


if BEDROCK_OK:
    cid1, cfg1, out1 = run_desk("CAIU9083321", "cma-cgm-ops")
    show_desk(out1, cfg1)
    print("\nledger:", json.dumps(EFFECT_LEDGER, indent=2)[:400])
else:
    print("skipped, no bedrock")

In [ ]:
# [needs bedrock] Run 2. Customs hold plus 1800 USD demurrage. The gate fires.
if BEDROCK_OK:
    cid2, cfg2, out2 = run_desk("MSCU7391045", "maersk-ops")
    show_desk(out2, cfg2)
    print("\neffects applied so far:", len(EFFECT_LEDGER), "  <- unchanged while paused")
else:
    print("skipped, no bedrock")

In [ ]:
# [needs bedrock] Run 2 continued. A human approves, hours later, from anywhere.
if BEDROCK_OK and "__interrupt__" in out2:
    resumed2 = release_desk.invoke(
        Command(resume={"approved": True, "approver": "ops_lead_2",
                        "note": "waiver agreed under dispute policy"}),
        cfg2,
    )
    print("resumed path:", " -> ".join(resumed2["path"]))
    print("approver    :", resumed2["approver"])
    print("ledger keys :", list(EFFECT_LEDGER))
    print()
    print("Note that intake, diagnose, contract and plan each appear once.")
    print("The swarm did not run again, so the resume cost no diagnosis tokens.")
else:
    print("skipped")

In [ ]:
# [needs bedrock] Run 3, denied. Run 4, rejected at intake before a single token.
if BEDROCK_OK:
    cid3, cfg3, out3 = run_desk("MSCU7391045", "maersk-ops")
    if "__interrupt__" in out3:
        out3 = release_desk.invoke(
            Command(resume={"approved": False, "approver": "ops_lead_2",
                            "note": "dispute unresolved, waiver not authorised"}), cfg3)
    print("DENIED  path:", " -> ".join(out3["path"]), "| reason:", out3["rejection_reason"])
    print("ledger unchanged:", len(EFFECT_LEDGER))

cid4, cfg4, out4 = run_desk("MSC7391045", "unknown-broker")
print("\nREJECTED path:", " -> ".join(out4["path"]))
print("reason       :", out4["rejection_reason"])
print("model calls  : zero. The cheapest check ran first.")

### The four runs, as a table you can put in front of an architect

| Run | Case | Path | Model calls | Human | Side effect |
|---|---|---|---|---|---|
| 1 | Equipment fault, no exposure | intake, diagnose, contract, plan, approval, apply_effects, audit | swarm, contract, planner | none, policy approved | release applied |
| 2 | Customs plus demurrage | stops after plan | swarm, contract, planner | approval requested | none while paused |
| 2b | Resumed, approved | approval, apply_effects, audit | zero extra | named approver stored | release applied once |
| 3 | Same, denied | ..., approval, rejected, audit | swarm, contract, planner | named approver stored | none |
| 4 | Bad container, wrong party | intake, rejected, audit | **zero** | none | none |

Run 4 is the one to point at. The cheapest possible check ran first and the expensive part of the system never woke up.

Run 2b is the one that would have been hardest to build by hand. The process that started the run was long gone. Everything needed to continue was in the checkpoint, keyed by the thread id, which was the correlation id all along.

## 10. Taking this to production

### Where state actually lives

`InMemorySaver` is for notebooks. It dies with the process, which is fine here and fatal anywhere else.

| Checkpointer | Install | Use for |
|---|---|---|
| `InMemorySaver` | built in | notebooks, tests |
| `SqliteSaver` | `langgraph-checkpoint-sqlite` | single-process apps, local development |
| `PostgresSaver` | `langgraph-checkpoint-postgres` | the normal production answer |
| LangGraph Platform | hosted | when you want the server, queueing and a built-in approval API |

Swapping is a one-line change at `compile()`. Nothing else in this notebook moves, which is a good reason to keep graph construction in a factory function rather than at module level.

### Thread ids

The thread id is your unit of state and your unit of memory. Get it wrong in the two obvious ways and you get two very different bugs.

| Mistake | What happens |
|---|---|
| A fresh thread id per request for the same case | The approval you are resuming belongs to a thread nobody will ever call again |
| One shared thread id across users | One customer's history leaks into another's run |

Make it the correlation id, put the same string in your logs, and both problems disappear.

### Things to set on purpose, not by accident

| Setting | Default | Why you should choose it yourself |
|---|---|---|
| `recursion_limit` | 25 | It is the only bound on a cyclic graph, and 25 is not a considered number for your workload |
| `RetryPolicy` | none | Bedrock throttling is normal. A node-level retry is one argument |
| `retry_on` | connection errors | It will not retry your own exceptions unless you list them |
| `interrupt_before` | none | Useful in development to pause anywhere without editing node code |
| Node `timeout` | none | A hung tool call otherwise hangs the run |

### Observability

Every model call, tool call and node in this notebook is already instrumented. Set four environment variables and the traces appear in LangSmith.

```bash
export LANGCHAIN_TRACING_V2=true
export LANGCHAIN_API_KEY=...
export LANGCHAIN_PROJECT=pierpoint-release-desk
```

If you would rather not send data to a hosted service, the same information is available locally.

| Signal | Where to get it |
|---|---|
| Node-by-node updates as they happen | `graph.stream(..., stream_mode="updates")` |
| Every state version, newest first | `graph.get_state_history(cfg)` |
| What is paused and what runs next | `graph.get_state(cfg).next` |
| Token usage | `response.usage_metadata` on any model reply |
| Custom events from inside a node | a `StreamWriter` argument, surfaced with `stream_mode="custom"` |

### The five alarms worth having

| Alarm | Signal |
|---|---|
| Recursion cap hits | `GraphRecursionError` rate |
| Approval backlog age | oldest thread sitting at `next == ("approval",)` |
| Swarm hop count at p95 | if it equals the cap, the cap is producing your answer |
| Idempotency collisions | how often `apply_effects` finds the key already there |
| Cost per request | summed `usage_metadata`, keyed by correlation id |

In [ ]:
# The test suite. No model calls, no AWS, no network. Run it in CI on every change.

# 1. Routing functions are plain Python, so test them like plain Python.
assert desk_after_intake({"problems": []}) == "diagnose"
assert desk_after_intake({"problems": ["bad id"]}) == "rejected"
assert desk_after_approval({"approved": True}) == "apply_effects"
assert desk_after_approval({"approved": False}) == "rejected"

# 2. The invariant gate.
assert desk_intake({"container_id": "MSCU7391045", "party": "maersk-ops"})["problems"] == []
assert desk_intake({"container_id": "MSC7391045", "party": "maersk-ops"})["problems"]
assert desk_intake({"container_id": "MSCU7391045", "party": "nobody"})["problems"]

# 3. Idempotency: applying twice must not write twice.
EFFECT_LEDGER.clear()
probe = {"correlation_id": "ci-1", "container_id": "TCLU1234567",
         "plan": {"steps": []}, "approver": "policy"}
desk_apply(probe)
desk_apply(probe)
assert len(EFFECT_LEDGER) == 1, EFFECT_LEDGER

# 4. The money rule, which is the one that protects real money.
assert _to_money(None) is None                       # missing stays unknown
assert _to_money("about 1,800 USD") == 1800.0        # prose still parses
assert _to_money(-50) is None                        # negative cannot slip under a threshold
assert RemediationPlan().financial_exposure_usd is None

# 5. The approval policy. Unknown exposure must never auto-approve.
assert approval_required(None, 500.0) is True        # missing number, ask a human
assert approval_required(1800.0, 500.0) is True
assert approval_required(120.0, 500.0) is False
assert approval_required(500.0, 500.0) is False      # the boundary is inclusive

# 6. Contracts absorb sloppy model output instead of raising.
assert DiagnosisReport(**{"findings": None, "unresolved": None}).findings == []
assert Finding(**{"blocking": "unclear"}).blocking is True

# 7. The graph is wired the way the diagram says.
nodes = set(release_desk.get_graph().nodes)
assert {"intake", "diagnose", "contract", "plan", "approval", "apply_effects", "rejected",
        "audit"} <= nodes, nodes

# 8. The gate really pauses. A two-node graph is enough to prove it, and it needs no model.
class _GateOnly(TypedDict):
    container_id: str
    plan: dict
    approved: bool
    approver: str
    rejection_reason: str
    path: Annotated[list[str], operator.add]

_g = StateGraph(_GateOnly)
_g.add_node("approval", desk_approval)
_g.add_node("apply_effects", lambda s: {"path": ["apply_effects"]})
_g.add_node("rejected", lambda s: {"path": ["rejected"]})
_g.add_edge(START, "approval")
_g.add_conditional_edges("approval", desk_after_approval, ["apply_effects", "rejected"])
_g.add_edge("apply_effects", END)
_g.add_edge("rejected", END)
_gate = _g.compile(checkpointer=InMemorySaver())

_blank = {"container_id": "TCLU1234567", "approved": False, "approver": "",
          "rejection_reason": "", "path": []}
_cfg = {"configurable": {"thread_id": "ci-gate-unknown"}}
_paused = _gate.invoke({**_blank, "plan": {"financial_exposure_usd": None}}, _cfg)
assert "__interrupt__" in _paused, "unknown exposure did not pause, which is the bug this test exists for"
assert _paused["__interrupt__"][0].value["exposure_known"] is False

_done = _gate.invoke(Command(resume={"approved": False, "approver": "ci", "note": "no"}), _cfg)
assert _done["path"] == ["approval", "rejected"], _done["path"]

_cfg2 = {"configurable": {"thread_id": "ci-gate-low"}}
_low = _gate.invoke({**_blank, "plan": {"financial_exposure_usd": 120.0}}, _cfg2)
assert "__interrupt__" not in _low and _low["path"] == ["approval", "apply_effects"], _low["path"]

print("all 8 groups passed: 0 tokens, 0 AWS calls")
print()
print("This is the part people assume you cannot test in an agentic system.")
print("Everything above is your control flow, and none of it needed a model.")

## 11. What we did

Start to finish, in one paragraph: a chain handled the straight line, then fell over the moment the work needed a branch, a loop, a pause and a memory. A StateGraph gave us all four from one decision, which was compiling with a checkpointer. A swarm handled the one stage where the next step genuinely could not be named in advance, and then the swarm went inside the graph rather than beside it, with a typed contract node between them.

### The rules worth keeping

1. If you can name the next step at design time, an LLM must not be the thing that picks it.
2. If a step writes, it does not get to decide whether it runs.
3. Prose never crosses a node boundary. Types do.
4. Coerce where the model is merely sloppy, fail closed where money or safety depends on the field. Unknown must never look like zero.
5. `interrupt()` goes at the top of a node that does nothing else, because that node will run twice.
6. A denied approval is a destination, so it gets an edge, not an exception.
7. Every side effect gets an idempotency key before it gets a retry policy.
8. The thread id is the correlation id. Same string in the graph and in the logs.

### The LangGraph pieces, and what each one is for

| Piece | Use it when |
|---|---|
| `StateGraph` plus `TypedDict` | The workflow has branches, loops, or state that outlives one call |
| Reducer on a key | More than one node writes to that key, especially in parallel |
| `add_conditional_edges` | A Python function should choose the path |
| Checkpointer plus `thread_id` | You need durability, memory, or the ability to pause |
| `interrupt()` and `Command(resume=)` | A human must decide before the run continues |
| `Command(goto=, graph=PARENT)` | Control moves between agents at runtime |
| `Send` | The number of parallel branches is known only at runtime |
| `recursion_limit` | Always. Any graph that can cycle |
| `RetryPolicy` | The node calls something that fails transiently |
| `create_agent` | You want a tool loop and do not want to draw one |
| `create_swarm` | You want the standard handoff behaviour today |

### What this notebook deliberately left out

- **Long-term memory across threads.** The `Store` interface handles facts that outlive a single case, and none of that is here.
- **Retrieval.** No vector store, no documents. The specialists know only what their tools return.
- **Evaluation.** Nothing here measures whether the diagnosis was *correct*, only that the system behaved lawfully. Those are different problems and the second one does not imply the first.
- **Streaming to a UI.** `stream_mode` is mentioned and not built out.

### Three things to try on your own work

1. Take one workflow you own and fill in the six-stage table from Part 1. Count the rows where you genuinely cannot name the next step. Most teams find zero, which means they need a graph and no swarm at all.
2. Move one invariant out of a prompt and into a routing function. Then write the test that would have caught the old version.
3. Add a second approval threshold with a different approver group. If your gate is still one node afterwards, the design is holding. If it has quietly become a rules engine, that is worth knowing early.